# APIs for data retrieval

An Application Programming Interface, commonly known as API, is a set of protocols, routines, and tools for building software applications. APIs allow different software systems to communicate with each other and exchange data in a standardized and efficient way.

APIs for retrieving data enable developers to access and extract information from various sources such as databases, web services, or applications. These APIs often provide a structured and consistent way of accessing data, making it easier for developers to consume and use the data in their applications.

APIs for retrieving data can be used for a variety of purposes, such as gathering information for business intelligence, analyzing user behavior, or integrating data from different sources into a single application. These APIs often use standardized data formats, such as JSON or XML, to ensure compatibility and interoperability between different systems.

As the amount of data available online continues to grow, APIs for retrieving data have become an essential tool for developers to access and analyze this data. By leveraging these APIs, developers can quickly and easily retrieve the data they need, without having to manually extract and process it themselves.

### API types
There are several types of APIs available, but two of the most commonly used types are REST API and HTTP API.

- REST API (Representational State Transfer API):
REST stands for Representational State Transfer, and it's a set of architectural principles for building web services. A REST API is a type of web service that follows the REST architecture principles. REST APIs use HTTP methods (GET, POST, PUT, DELETE, etc.) to access and manipulate resources, which are identified by URIs (Uniform Resource Identifiers). REST APIs typically return data in JSON or XML format and are widely used for building web and mobile applications.


- HTTP API (Hypertext Transfer Protocol API):
HTTP stands for Hypertext Transfer Protocol, which is the protocol used for transferring data over the World Wide Web. An HTTP API is a type of web service that uses HTTP methods to access and manipulate resources. An HTTP API can be RESTful, but it doesn't have to be. HTTP APIs are often used for simple operations like CRUD (Create, Read, Update, Delete) on resources and return data in JSON or XML format.


Other types of APIs include SOAP (Simple Object Access Protocol), GraphQL, and WebSockets. SOAP is an older protocol used for building web services, while GraphQL is a newer API technology that allows clients to specify the data they need and receive it in a single request. WebSockets are used for real-time, two-way communication between a client and a server.

### python and APIs

Python is a popular programming language that provides powerful tools for interacting with APIs. Here are the steps to interact with APIs using Python:

1. Import the necessary libraries: Python has several libraries that make it easy to interact with APIs, including requests, json, and urllib. Before making any API requests, you need to import the appropriate libraries.

2. Find the API endpoint: The endpoint is the URL that you will use to send your API requests. It's essential to understand the API documentation to find the correct endpoint for the specific data you want to retrieve.

3. Send a request: Once you have the endpoint, you can use Python's requests library to send an HTTP request to the API endpoint. The requests library has several methods for sending different types of HTTP requests, including GET, POST, PUT, DELETE, and more.

4. Parse the response: The API response will typically be in JSON format. Python's json library can be used to parse the JSON data and convert it into a Python dictionary that you can easily work with in your code.

5. Extract the data: Once you have the API response in a Python dictionary, you can extract the data you need and use it in your application.

Python's ease of use and powerful libraries make it an excellent language for interacting with APIs. With just a few lines of code, you can send requests to APIs, parse the response data, and extract the information you need to build powerful applications.

### Using the request package

To do this, we need to know how to send requests first. We will use an amazing package called [`requests`](http://docs.python-requests.org/en/master/). If you do not have it installed, please install it using e.g. `poetry add` (in your command prompt or terminal):


```$ poetry add requests``` or ```$ pip install requests```


In [ ]:
import requests # library for making HTTP requests
import pandas as pd # library for data analysis
import datetime as dt # library for handling date and time objects

###### open DMI weather data 

Go to the [documentation](https://opendatadocs.dmi.govcloud.dk/en/DMIOpenData)

Using weather as an example, we should first know what is the request URL (where the request goes to), with what parameters(e.g., API key and stationID). In our case, we know that our API key and the stationId to query so we can do the following.

You will have to create a user and retrieve an API key for the API you want to use [how to](https://opendatadocs.dmi.govcloud.dk/Authentication)

I have saved my API key in an file ```.env```
    
    api_key = your_api_key 

Specifically we will look at [Meteorological Observation](https://opendatadocs.dmi.govcloud.dk/en/APIs/Meteorological_Observation_API) 

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.environ['api_key']

In [ ]:
# Alternative: Read the keys and tokens into a dictionary
# my_dict = {}

# with open("./.env", "r") as f:
#     for line in f:
#         key,val = line.split('=')
#         my_dict[key.strip()] = val.strip()
        
# api_key = my_dict['api_key']

In [ ]:
stationId = '06074' # list of stationId: https://confluence.govcloud.dk/pages/viewpage.action?pageId=41717704

In [ ]:
DMI_URL = 'https://dmigw.govcloud.dk/v2/metObs/collections/station/items'
r = requests.get(DMI_URL, params={'api-key': api_key, "stationId": stationId}) # Issues a HTTP GET request
r.url # `requests` help us encode the URL in the correct format

In [ ]:
r.status_code # 200 means success

In [ ]:
station = r.json()
station

In [ ]:
DMI_URL = 'https://dmigw.govcloud.dk/v2/metObs/collections/observation/items'
r = requests.get(DMI_URL, params={'api-key': api_key, "stationId": '06030', 'period': 'latest-day', 'parameterId': 'temp_dry'}) # Issues a HTTP GET request
print(r)

In [ ]:
dmi = r.json()  # Extract JSON data
dmi  # Print the keys of the JSON dictionary

JSON object will be converted into a `dict` type, which is the data structure in Python holding key value pairs. To access certain values, we just access them like a `dict`.

In [ ]:
dmi['features']

In [ ]:
dmi['features'][0]

In [ ]:
dmi['features'][0]['properties']

In [ ]:
dmi['features'][0]['properties']['value']

In [ ]:
for feature in dmi['features']:
    for key, value in feature['properties'].items():
         print(key, value)

Now it gets interesting, as we can put the values into a dataframe (more on dataframes later).

In [ ]:
import pandas as pd

lst = []

for values in dmi['features']:
    lst.append(pd.DataFrame.from_dict(values['properties'], orient='index').transpose())

In [ ]:
df = pd.concat(lst).reset_index()

In [ ]:
df

### HTTP API

An example from Open Data DK 
https://www.opendata.dk/syddjurs-kommune/indeklima-administrationsbygningen-i-hornslet1

In [ ]:
import requests
import pandas as pd

In [ ]:
r = requests.get('https://os2iot-backend.gtm.context.kmd.dk/api/v1/open-data-dk-sharing/22/data/22')
r.json()

In [ ]:
schema = r.json()[0][0].keys()
df = pd.DataFrame(columns=schema)
df['time'] = []

for t in r.json():
    for i in t:
        df.loc[len(df.index)] = [
            i['id'], 
            i['type'], 
            i['name']['value'], 
            i['temperature']['value'], 
            i['humidity']['value'],  
            i['light_level']['value'],  
            i['motion']['value'], 
            i['co2']['value'],  
            i['location']['value']['coordinates'],
            i['temperature']['observedAt'],
        ]

In [ ]:
df

An example from Open Data DK 
https://www.opendata.dk/city-of-aarhus/transaktionsdata-fra-aarhus-kommunes-biblioteker

In [ ]:
url = 'https://admin.opendata.dk/api/3/action/datastore_search?resource_id=5b9b00f9-543e-4ac0-994c-dbbc8b38e7e5'
r = requests.get(url)
r.json()

In [ ]:
# Using sql in query for filtering
sql_url = 'https://admin.opendata.dk/api/3/action/datastore_search_sql?sql=SELECT * from "5b9b00f9-543e-4ac0-994c-dbbc8b38e7e5" WHERE id=3'
r = requests.get(sql_url)
r.json()

## API Best Practices

Working with APIs requires careful handling of errors, timeouts, and security. Here are essential best practices.

### Error Handling

Always handle potential errors when working with APIs. Network issues, server errors, and invalid responses can occur.

In [ ]:
import requests
import time

def fetch_data_safe(url, params=None, max_retries=3):
    """
    Safely fetch data from an API with error handling and retries.
    
    Args:
        url: API endpoint URL
        params: Optional parameters
        max_retries: Maximum number of retry attempts
        
    Returns:
        JSON response or None if all attempts fail
    """
    for attempt in range(max_retries):
        try:
            # Set timeout to prevent hanging
            response = requests.get(url, params=params, timeout=10)
            
            # Check if request was successful
            response.raise_for_status()  # Raises HTTPError for bad status codes
            
            return response.json()
            
        except requests.exceptions.Timeout:
            print(f'Timeout on attempt {attempt + 1}/{max_retries}')
            
        except requests.exceptions.ConnectionError:
            print(f'Connection error on attempt {attempt + 1}/{max_retries}')
            
        except requests.exceptions.HTTPError as e:
            print(f'HTTP error: {e}')
            if response.status_code == 404:
                print('Resource not found')
                return None
            elif response.status_code >= 500:
                print('Server error, retrying...')
            else:
                return None
                
        except requests.exceptions.JSONDecodeError:
            print('Invalid JSON response')
            return None
            
        except Exception as e:
            print(f'Unexpected error: {e}')
            return None
        
        # Wait before retrying
        if attempt < max_retries - 1:
            wait_time = 2 ** attempt  # Exponential backoff
            print(f'Waiting {wait_time} seconds before retry...')
            time.sleep(wait_time)
    
    print('All retry attempts failed')
    return None

# Example usage
# data = fetch_data_safe('https://api.example.com/data')

### Secure API Key Management

**Never** hard-code API keys in your notebooks or commit them to version control!

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# ✅ GOOD: Read API key from environment variable
api_key = os.getenv('API_KEY')

# Alternative: Read from .env file manually
def load_env_file(filepath='.env'):
    """Load environment variables from .env file."""
    env_vars = {}
    try:
        with open(filepath, 'r') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('#'):
                    key, value = line.split('=', 1)
                    env_vars[key.strip()] = value.strip()
    except FileNotFoundError:
        print(f'Warning: {filepath} not found!')
    return env_vars

# Usage
# env_vars = load_env_file()
# api_key = env_vars.get('API_KEY')

### Rate Limiting

Many APIs have rate limits. Respect them to avoid being blocked!

In [ ]:
import time
from datetime import datetime

class RateLimiter:
    """Simple rate limiter for API calls."""
    
    def __init__(self, calls_per_minute=60):
        self.calls_per_minute = calls_per_minute
        self.min_interval = 60.0 / calls_per_minute
        self.last_call = None
    
    def wait_if_needed(self):
        """Wait if necessary to respect rate limit."""
        if self.last_call is not None:
            elapsed = time.time() - self.last_call
            if elapsed < self.min_interval:
                wait_time = self.min_interval - elapsed
                print(f'Rate limiting: waiting {wait_time:.2f} seconds...')
                time.sleep(wait_time)
        
        self.last_call = time.time()

# Example usage
limiter = RateLimiter(calls_per_minute=30)  # Max 30 calls per minute

def fetch_with_rate_limit(url):
    """Fetch data while respecting rate limits."""
    limiter.wait_if_needed()
    response = requests.get(url, timeout=10)
    return response.json()

# Make multiple requests
# for i in range(5):
#     data = fetch_with_rate_limit(f'https://api.example.com/item/{i}')
#     print(f'Fetched item {i}')

### Setting Timeouts

Always set timeouts to prevent your program from hanging indefinitely.

In [ ]:
# ✅ GOOD: With timeout
try:
    response = requests.get('https://api.example.com/data', timeout=10)
except requests.exceptions.Timeout:
    print('Request timed out after 10 seconds')

# ❌ BAD: No timeout (can hang forever)
# response = requests.get('https://api.example.com/data')

# You can also set separate connect and read timeouts
# response = requests.get(url, timeout=(3, 10))  # 3s connect, 10s read

### Checking Response Status

Always verify the response was successful before processing.

In [ ]:
def check_response_status(response):
    """Check and handle different HTTP status codes."""
    
    if response.status_code == 200:
        print('✓ Success')
        return True
        
    elif response.status_code == 400:
        print('✗ Bad Request - check your parameters')
        
    elif response.status_code == 401:
        print('✗ Unauthorized - check your API key')
        
    elif response.status_code == 403:
        print('✗ Forbidden - you don\'t have permission')
        
    elif response.status_code == 404:
        print('✗ Not Found - endpoint doesn\'t exist')
        
    elif response.status_code == 429:
        print('✗ Too Many Requests - you\'re being rate limited')
        
    elif response.status_code >= 500:
        print('✗ Server Error - try again later')
        
    else:
        print(f'✗ Unexpected status code: {response.status_code}')
    
    return False

# Example
# response = requests.get('https://api.example.com/data')
# if check_response_status(response):
#     data = response.json()

### Complete Example: Robust API Call

In [ ]:
import requests
import time
import os
from typing import Optional, Dict, Any

def make_api_call(
    url: str,
    params: Optional[Dict[str, Any]] = None,
    headers: Optional[Dict[str, str]] = None,
    timeout: int = 10,
    max_retries: int = 3
) -> Optional[Dict]:
    """
    Make a robust API call with best practices.
    
    Args:
        url: API endpoint URL
        params: Query parameters
        headers: HTTP headers (including auth)
        timeout: Request timeout in seconds
        max_retries: Maximum retry attempts
        
    Returns:
        JSON response or None if failed
    """
    
    for attempt in range(max_retries):
        try:
            response = requests.get(
                url,
                params=params,
                headers=headers,
                timeout=timeout
            )
            
            # Check status
            if response.status_code == 200:
                return response.json()
                
            elif response.status_code == 429:
                # Rate limited - wait longer
                wait_time = int(response.headers.get('Retry-After', 60))
                print(f'Rate limited. Waiting {wait_time}s...')
                time.sleep(wait_time)
                continue
                
            elif response.status_code >= 500:
                # Server error - retry with backoff
                wait_time = 2 ** attempt
                print(f'Server error. Retrying in {wait_time}s...')
                time.sleep(wait_time)
                continue
                
            else:
                # Client error - don't retry
                print(f'Request failed: {response.status_code}')
                print(response.text)
                return None
                
        except requests.exceptions.Timeout:
            print(f'Timeout on attempt {attempt + 1}')
            
        except requests.exceptions.RequestException as e:
            print(f'Request error: {e}')
            
        if attempt < max_retries - 1:
            time.sleep(1)
    
    return None

# Example usage
# api_key = os.getenv('API_KEY')
# headers = {'Authorization': f'Bearer {api_key}'}
# data = make_api_call(
#     'https://api.example.com/data',
#     params={'limit': 100},
#     headers=headers
# )

### Summary of Best Practices

1. **Always handle errors** with try-except blocks
2. **Set timeouts** to prevent hanging
3. **Store API keys securely** in environment variables
4. **Respect rate limits** and implement backoff strategies
5. **Check response status** before processing data
6. **Implement retry logic** for transient errors
7. **Log errors** for debugging
8. **Use sessions** for multiple requests to the same host
9. **Validate responses** before using the data
10. **Document your API interactions** for other developers

return to [overview](../00_overview.ipynb)